# Playground

Front-facing notebook for rerunning the benchmark with your own choices:
- leverage bounds
- asset count and tickers
- NN method family
- horizon / target multiplier
- seeds and training budget

Edit the config cell, run the notebook, and then inspect the resulting tables and plots.

In [ ]:
from pathlib import Path
import sys
import math

ROOT = Path.cwd().resolve()
while not (ROOT / 'comparisons').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    import pandas as pd
except ImportError:
    pd = None

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

from comparisons.core.notebook_api import (
    load_summary_table,
    list_available_runs,
    load_fd_policy_bundle,
    load_nn_model_bundle,
)
from comparisons.core.io import load_run_result
from comparisons.core.config import BenchmarkConfig
from comparisons.core.evaluation import run_real_data_portfolio_comparison
from real_data_loader import load_portfolio

In [ ]:
RESULTS_DIR = ROOT / 'comparisons' / 'results'

N_ASSETS = 5
START_DATE = '2020-01-01'
END_DATE = '2020-12-31'
INITIAL_WEALTH_LEVELS = [1.0]
TARGET_MULTIPLIER = 1.10
RANDOM_SEEDS = [1]

INCLUDE_FD_BENCHMARK = True
INCLUDE_MERTON_BENCHMARK = True
INCLUDE_NN = True

NN_METHODS = [
    'nn_mlp_small',
    'nn_mlp_deep',
    'deep_bsde',
    'pinn',
    'actor_critic',
    'lstm',
    'transformer',
]

FD_NW = 120
FD_NT = 80
NN_ITERS = 2
NN_PATHS = 32
NN_STEPS = 6

WEIGHT_LOWER_BOUND = -5.0
WEIGHT_UPPER_BOUND = 3.0
MAX_LONG_LEVERAGE = 3.0
MAX_SHORT_LEVERAGE = 5.0

## Optional Custom Tickers

If you want custom tickers, edit this list and use it as a guide for a custom market loader later.
Right now the benchmark uses the built-in 5/10/20 ETF portfolios from `real_data_loader.py`.

In [ ]:
CUSTOM_TICKERS = None
print('Custom tickers:', CUSTOM_TICKERS)
print('Built-in portfolio size:', N_ASSETS)

In [ ]:
config = BenchmarkConfig(
    n_assets_list=[N_ASSETS],
    start_date=START_DATE,
    end_date=END_DATE,
    initial_wealth_levels=INITIAL_WEALTH_LEVELS,
    target_multiplier=TARGET_MULTIPLIER,
    random_seeds=RANDOM_SEEDS,
    results_dir=RESULTS_DIR,
    include_fd_benchmark=INCLUDE_FD_BENCHMARK,
    include_merton_benchmark=INCLUDE_MERTON_BENCHMARK,
    include_nn=INCLUDE_NN,
    nn_architectures=NN_METHODS,
    fd_nw=FD_NW,
    fd_nt=FD_NT,
    nn_iters=NN_ITERS,
    nn_paths=NN_PATHS,
    nn_steps=NN_STEPS,
    weight_lower_bound=WEIGHT_LOWER_BOUND,
    weight_upper_bound=WEIGHT_UPPER_BOUND,
    max_long_leverage=MAX_LONG_LEVERAGE,
    max_short_leverage=MAX_SHORT_LEVERAGE,
)
config

In [ ]:
outputs = run_real_data_portfolio_comparison(config)
print('runs:', len(outputs['results']))
print('plots:', outputs['plots'])
print('tables:', sorted(outputs['tables'].keys()))

In [ ]:
main_results = load_summary_table('main_results', results_dir=RESULTS_DIR)
aggregated_results = load_summary_table('aggregated_results', results_dir=RESULTS_DIR)
neural_results = load_summary_table('neural_family_results', results_dir=RESULTS_DIR)

if pd is not None:
    agg_df = pd.DataFrame(aggregated_results)
    display(agg_df[agg_df['n_assets'].astype(int) == N_ASSETS])
else:
    print(aggregated_results[:5])

In [ ]:
fd_bundle = load_fd_policy_bundle(n_assets=N_ASSETS, seed=RANDOM_SEEDS[0], initial_wealth=INITIAL_WEALTH_LEVELS[0], results_dir=RESULTS_DIR)
fd_policy = fd_bundle['policy']
print('FD policy sample:', fd_policy(1.0, 0.5))

In [ ]:
nn_bundles = {
    name: load_nn_model_bundle(name, n_assets=N_ASSETS, seed=RANDOM_SEEDS[0], initial_wealth=INITIAL_WEALTH_LEVELS[0], results_dir=RESULTS_DIR)
    for name in NN_METHODS
}
for name, bundle in nn_bundles.items():
    w = bundle['weights_fn'](1.0, TARGET_MULTIPLIER)
    print(name, np.round(w, 4))

In [ ]:
plot_names = [
    'fd_vs_neural_target_hit_rate.png',
    'fd_vs_neural_mean_terminal_wealth.png',
    'fd_vs_neural_runtime.png',
    'neural_risk_vs_performance.png',
]
for plot_name in plot_names:
    display(Markdown(f'### {plot_name}'))
    display(Image(filename=str(RESULTS_DIR / 'plots' / plot_name)))